In [4]:
# Install (usually already installed in Colab)
!pip install pandas numpy scikit-learn tensorflow

# Imports
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [5]:
import pandas as pd
#Reading the dataset, skipping bad lines to handle parsing errors with the Python engine
df = pd.read_csv('/content/train.csv', encoding='latin1', on_bad_lines='skip', engine='python')

In [6]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [7]:
import re

# Keep required columns
df = df[['comment_text', 'toxic']].dropna()

# Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['comment_text'].apply(clean_text)

df.head()

,comment_text,toxic,clean_text
0,Explanation\nWhy the edits made under my usern...,0,explanation why the edits made under my userna...
1,D'aww! He matches this background colour I'm s...,0,d aww he matches this background colour i m se...
2,"Hey man, I'm really not trying to edit war. It...",0,hey man i m really not trying to edit war it s...
3,"""\nMore\nI can't make any real suggestions on ...",0,more i can t make any real suggestions on impr...
4,"You, sir, are my hero. Any chance you remember...",0,you sir are my hero any chance you remember wh...


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['clean_text'])

X = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(X, maxlen=max_len)

y = df['toxic']

print("Shape:", X.shape)

Shape: (11939, 100)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))

history = model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights
)

Epoch 1/3
239/239 ━━━━━━━━━━━━━━━━━━━━ 38s 142ms/step - accuracy: 0.7941 - loss: 0.4980 - val_accuracy: 0.9173 - val_loss: 0.2611
Epoch 2/3
239/239 ━━━━━━━━━━━━━━━━━━━━ 30s 96ms/step - accuracy: 0.9190 - loss: 0.2232 - val_accuracy: 0.9367 - val_loss: 0.1729
Epoch 3/3
239/239 ━━━━━━━━━━━━━━━━━━━━ 41s 96ms/step - accuracy: 0.9658 - loss: 0.0988 - val_accuracy: 0.9252 - val_loss: 0.1879


In [12]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")
print(classification_report(y_test, y_pred))

75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step
              precision    recall  f1-score   support

           0       0.96      0.95      0.96      2152
           1       0.59      0.68      0.63       236

    accuracy                           0.92      2388
   macro avg       0.78      0.81      0.79      2388
weighted avg       0.93      0.92      0.92      2388



In [13]:
import pickle
# Save model
model.save("toxicity_model.h5")

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Saved successfully!")

Saved successfully!


In [14]:
def predict_toxicity(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    prediction = model.predict(padded)[0][0]
    return prediction

# Test
comment = "You are a horrible person!"
score = predict_toxicity(comment)

print("Score:", score)
print("Toxic" if score > 0.5 else "Non-Toxic")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Score: 0.767948
Toxic


In [15]:
print(df['toxic'].value_counts())

toxic
0    10811
1     1128
Name: count, dtype: int64


In [18]:
import os

# ── Step 1: Install dependencies ──────────────────────────────────────────
!pip install streamlit pyngrok -q

# ── Step 2: Write app.py ──────────────────────────────────────────────────
app_code = '''
import streamlit as st
import numpy as np
import re
import pickle
import os
import pandas as pd

st.set_page_config(page_title="Comment Toxicity Detector", page_icon="💬", layout="wide")

MAX_LEN = 100

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

@st.cache_resource
def load_artifacts():
    from tensorflow.keras.models import load_model
    model = load_model("/content/toxicity_model.h5")
    with open("/content/tokenizer.pkl", "rb") as f:
        tokenizer = pickle.load(f)
    return model, tokenizer

def predict(text, model, tokenizer):
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    seq = tokenizer.texts_to_sequences([clean_text(text)])
    padded = pad_sequences(seq, maxlen=MAX_LEN)
    return float(model.predict(padded, verbose=0)[0][0])

st.markdown("""
<style>
body { background-color: #0e1117; }
.title { text-align:center; font-size:2.5rem; font-weight:800;
         background: linear-gradient(90deg,#ff4b4b,#ff914d);
         -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.subtitle { text-align:center; color:#aaa; margin-bottom:2rem; }
.toxic-box { background:#2d0000; border-left:5px solid #ff4b4b;
             padding:1rem; border-radius:8px; }
.safe-box  { background:#002d00; border-left:5px solid #00c853;
             padding:1rem; border-radius:8px; }
.word-toxic { background:#ff4b4b; color:white; padding:2px 8px;
              border-radius:4px; margin:2px; display:inline-block; font-weight:600; }
.word-safe  { color:#ddd; margin:2px; display:inline-block; }
</style>
""", unsafe_allow_html=True)

st.markdown('<div class="title">💬 Comment Toxicity Detector</div>', unsafe_allow_html=True)
st.markdown('<div class="subtitle">Detect toxic language instantly using Deep Learning</div>', unsafe_allow_html=True)

model, tokenizer = load_artifacts()

tab1, tab2, tab3, tab4, tab5 = st.tabs(["🔍 Single Comment", "📁 Bulk CSV Upload", "📊 Data Insights", "📈 Model Performance", "🧪 Sample Test Cases"])

with tab1:
    comment = st.text_area("Enter a comment:", placeholder="Type something here...", height=150)
    if st.button("🔍 Analyse", use_container_width=True):
        if comment.strip() == "":
            st.warning("Please enter a comment.")
        else:
            score = predict(comment, model, tokenizer)
            label = "🚨 TOXIC" if score > 0.5 else "✅ SAFE"
            box   = "toxic-box" if score > 0.5 else "safe-box"
            col1, col2 = st.columns(2)
            col1.metric("Toxicity Score", f"{score:.2%}")
            col2.metric("Verdict", label)
            st.progress(score)
            st.markdown("#### Word Highlights")
            words = comment.split()
            highlighted = ""
            for word in words:
                w_score = predict(word, model, tokenizer)
                if w_score > 0.5:
                    highlighted += f' <span class="word-toxic">{word}</span> '
                else:
                    highlighted += f' <span class="word-safe">{word}</span> '
            st.markdown(highlighted, unsafe_allow_html=True)
            st.markdown(f'<div class="{box}"><b>{label}</b> — Raw score: {score:.4f}</div>', unsafe_allow_html=True)

with tab2:
    st.markdown("Upload a CSV file with a **`comment_text`** column.")
    uploaded = st.file_uploader("Choose CSV", type=["csv"])
    if uploaded:
        df = pd.read_csv(uploaded)
        if "comment_text" not in df.columns:
            st.error("CSV must have a 'comment_text' column.")
        else:
            st.write(f"**{len(df)} comments found.** Running predictions...")
            progress = st.progress(0)
            scores, labels = [], []
            for i, row in enumerate(df["comment_text"]):
                s = predict(str(row), model, tokenizer)
                scores.append(round(s, 4))
                labels.append("Toxic" if s > 0.5 else "Safe")
                progress.progress((i + 1) / len(df))
            df["toxicity_score"] = scores
            df["label"] = labels
            toxic_count = labels.count("Toxic")
            safe_count  = labels.count("Safe")
            c1, c2, c3 = st.columns(3)
            c1.metric("Total Comments", len(df))
            c2.metric("🚨 Toxic", toxic_count)
            c3.metric("✅ Safe", safe_count)
            st.dataframe(
                df[["comment_text", "toxicity_score", "label"]].style.applymap(
                    lambda v: "background-color:#2d0000;color:#ff4b4b" if v == "Toxic"
                              else ("background-color:#002d00;color:#00c853" if v == "Safe" else ""),
                    subset=["label"]
                ),
                use_container_width=True
            )
            csv = df.to_csv(index=False).encode("utf-8")
            st.download_button("⬇️ Download Results", csv, "predictions.csv", "text/csv", use_container_width=True)

with tab3:
    st.header("📊 Data Insights")
    st.subheader("Class Distribution of Training Data")
    st.write("The training dataset used for this model had the following distribution of comments:")
    st.markdown("- **Non-Toxic (0):** 10811 comments")
    st.markdown("- **Toxic (1):** 1128 comments")
    st.write(f"Total comments in training data: {10811 + 1128}")
    st.info("This dataset exhibits a significant class imbalance, with far more non-toxic comments than toxic ones. The model was trained using class weighting to address this imbalance.")

with tab4:
    st.header("📈 Model Performance Metrics")
    st.subheader("Classification Report on Test Data")
    st.text("""
              precision    recall  f1-score   support\n
           0       0.96      0.95      0.96      2152
           1       0.59      0.68      0.63       236

    accuracy                           0.92      2388
   macro avg       0.78      0.81      0.79      2388
weighted avg       0.93      0.92      0.92      2388
""")
    st.info("The model demonstrates high precision and recall for non-toxic comments. For toxic comments, while recall is reasonable, precision is lower, indicating some false positives. The balanced F1-score for toxic comments is 0.63.")

with tab5:
    st.header("🧪 Sample Test Cases")
    sample_comments = [
        {"comment": "You are a wonderful person!", "expected": "Safe"},
        {"comment": "I hate you, you are so stupid!", "expected": "Toxic"},
        {"comment": "This is a great movie, loved it.", "expected": "Safe"},
        {"comment": "Shut up, you pathetic idiot.", "expected": "Toxic"},
        {"comment": "I disagree with your point.", "expected": "Safe"},
        {"comment": "Your work is terrible and uninspired.", "expected": "Toxic"},
        {"comment": "That was a constructive criticism, thank you.", "expected": "Safe"}
    ]

    st.markdown("Here are some predefined sample comments to test the model's predictions:")

    for i, sample in enumerate(sample_comments):
        st.subheader(f"Sample {i+1}:")
        st.code(sample['comment'])
        col1, col2, col3 = st.columns(3)
        col1.markdown(f"**Expected:** `{sample['expected']}`")
        score = predict(sample['comment'], model, tokenizer)
        label = "🚨 TOXIC" if score > 0.5 else "✅ SAFE"
        box_class = "toxic-box" if score > 0.5 else "safe-box"
        col2.markdown(f"**Predicted:** {label}")
        col3.markdown(f"**Toxicity Score:** `{score:.2%}`")
        st.progress(score)
        st.markdown(f'<div class="{box_class}"><b>{label}</b> — Raw score: {score:.4f}</div>', unsafe_allow_html=True)
        st.markdown("---")

'''

with open("/content/app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written!")

✅ app.py written!


In [19]:
# ── Kill existing processes ────────────────────────────────────────────────
import os, subprocess, time

os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(2)

# ── Start Streamlit ────────────────────────────────────────────────────────
subprocess.Popen(
    ["streamlit", "run", "/content/app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--server.enableXsrfProtection=false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)
print("✅ Streamlit started on port 8501")

# ── Get Colab public URL ───────────────────────────────────────────────────
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print("\n" + "="*50)
print(f"🚀 LIVE URL: {url}")
print("="*50)
print("\n⚠️  If the page looks blank, click the URL and")
print("   allow 3rd party cookies in your browser.")

✅ Streamlit started on port 8501

🚀 LIVE URL: https://8501-m-s-kkb-use1c0-2fuws6kg39y2l-c.us-east1-0.prod.colab.dev

⚠️  If the page looks blank, click the URL and
   allow 3rd party cookies in your browser.
